In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from pathlib import Path

In [2]:
path = Path('/kaggle/input/predict-podcast-new')
train = pd.read_csv(path/'train.csv')
test = pd.read_csv(path/'test.csv')
sub = pd.read_csv(path/'sample_submission.csv')
train.drop(columns=["id"],axis=1,inplace=True)

In [3]:
print("Check Out Train DaTA Null Values: ",train.isnull().sum())
print("#"*130)
print(f"Train Data Shape: {train.shape}")
print("#"*130)
print(f"Train Data INFO: {train.info()}")
print("#"*130)

Check Out Train DaTA Null Values:  Podcast_Name                        0
Episode_Title                       0
Episode_Length_minutes          87093
Genre                               0
Host_Popularity_percentage          0
Publication_Day                     0
Publication_Time                    0
Guest_Popularity_percentage    146030
Number_of_Ads                       1
Episode_Sentiment                   0
Listening_Time_minutes              0
dtype: int64
##################################################################################################################################
Train Data Shape: (750000, 11)
##################################################################################################################################
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Podcast

In [4]:
print("Check Out Test DaTA Null Values: ",test.isnull().sum())
print("#"*130)
print(f"Test Data Shape: {test.shape}")
print("#"*130)
print(f"Test Data INFO: {test.info()}")
print("#"*130)

Check Out Test DaTA Null Values:  id                                 0
Podcast_Name                       0
Episode_Title                      0
Episode_Length_minutes         28736
Genre                              0
Host_Popularity_percentage         0
Publication_Day                    0
Publication_Time                   0
Guest_Popularity_percentage    48832
Number_of_Ads                      0
Episode_Sentiment                  0
dtype: int64
##################################################################################################################################
Test Data Shape: (250000, 11)
##################################################################################################################################
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                  

In [5]:
def clean_data(df):
    df=df.copy()
    df['Episode_Length_minutes'] = df['Episode_Length_minutes'].fillna(df['Episode_Length_minutes'].mean())
    df["Guest_Popularity_percentage"] = df['Guest_Popularity_percentage'].fillna(df["Guest_Popularity_percentage"].mean())

    return df

train=clean_data(train)
test=clean_data(test)

In [6]:
train.head()

,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
0,Mystery Matters,Episode 98,64.504738,True Crime,74.81,Thursday,Night,52.236449,0.0,Positive,31.41998
1,Joke Junction,Episode 26,119.800000,Comedy,66.95,Saturday,Afternoon,75.950000,2.0,Negative,88.01241
2,Study Sessions,Episode 16,73.900000,Education,69.97,Tuesday,Evening,8.970000,0.0,Negative,44.92531
3,Digital Digest,Episode 45,67.170000,Technology,57.22,Monday,Morning,78.700000,2.0,Positive,46.27824
4,Mind & Body,Episode 86,110.510000,Health,80.07,Monday,Afternoon,58.680000,3.0,Neutral,75.61031


In [7]:
test.head()

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment
0,750000,Educational Nuggets,Episode 73,78.96,Education,38.11,Saturday,Evening,53.330000,1.0,Neutral
1,750001,Sound Waves,Episode 23,27.87,Music,71.29,Sunday,Morning,52.192796,0.0,Neutral
2,750002,Joke Junction,Episode 11,69.10,Comedy,67.89,Friday,Evening,97.510000,0.0,Positive
3,750003,Comedy Corner,Episode 73,115.39,Comedy,23.40,Sunday,Morning,51.750000,2.0,Positive
4,750004,Life Lessons,Episode 50,72.32,Lifestyle,58.10,Wednesday,Morning,11.300000,2.0,Neutral


In [8]:
cat_cols = ['Podcast_Name', 'Episode_Title', 'Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']
for c in cat_cols:
    train[c] = train[c].astype('category')
    test[c] = test[c].astype('category')


y = train['Listening_Time_minutes'].values
train = train.drop(['Listening_Time_minutes'], axis=1)
test.drop(columns=["id"],axis=1,inplace=True)

In [9]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(train))
pred_lgb = np.zeros(len(test))
oof_cb = np.zeros(len(train))
pred_cb = np.zeros(len(test))
oof_xgb = np.zeros(len(train))
pred_xgb = np.zeros(len(test))

for fold, (trn_idx, val_idx) in enumerate(kf.split(train)):
    X_tr, X_val = train.iloc[trn_idx], train.iloc[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]

    # === LightGBM ===
    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

    params_lgb = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.05,
        'num_leaves': 256,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'seed': 42
    }

    model_lgb = lgb.train(
        params_lgb,
        lgb_train,
        num_boost_round=1000,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )

    oof_lgb[val_idx] = model_lgb.predict(X_val)
    pred_lgb += model_lgb.predict(test) / 10

    # === CatBoost ===
    model_cb = cb.CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=10,
        loss_function='RMSE',
        random_seed=42,
        verbose=False,
        cat_features=cat_cols
    )

    model_cb.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100, verbose=False)
    oof_cb[val_idx] = model_cb.predict(X_val)
    pred_cb += model_cb.predict(test) / 10

    # === XGBoost ===
    model_xgb = xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=9,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        objective='reg:squarederror',
        random_state=42,
        enable_categorical=True,
        verbosity=0
    )

    model_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=100, verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_val)
    pred_xgb += model_xgb.predict(test) / 10

Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[998]	valid_0's rmse: 12.7891


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[991]	valid_0's rmse: 12.7573


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[999]	valid_0's rmse: 12.8094


...

/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 12.7477


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[988]	valid_0's rmse: 12.8549


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


In [10]:
oof_df = pd.DataFrame({'lgb': oof_lgb,'cb': oof_cb,'xgb': oof_xgb,'y': y})

def find_best_weights(df):
    best_rmse = 1e9
    best_w = None
    step = 0.01
    for w1 in np.arange(0.0, 1.01, step):
        for w2 in np.arange(0.0, 1.01 - w1, step):
            w3 = 1.0 - w1 - w2
            pred = w1 * df['lgb'] + w2 * df['cb'] + w3 * df['xgb']
            rmse = mean_squared_error(df['y'], pred, squared=False)
            if rmse < best_rmse:
                best_rmse = rmse
                best_w = (w1, w2, w3)
    return best_w

w_lgb, w_cb, w_xgb = find_best_weights(oof_df)
print(f"Best weights → LGB: {w_lgb:.3f} | CB: {w_cb:.3f} | XGB: {w_xgb:.3f}")

final_pred = w_lgb * pred_lgb + w_cb * pred_cb + w_xgb * pred_xgb

Best weights → LGB: 0.580 | CB: 0.110 | XGB: 0.310


In [11]:
sub['Listening_Time_minutes'] = final_pred
sub.to_csv('submission.csv', index=False)
sub.head()

,id,Listening_Time_minutes
0,750000,52.407142
1,750001,18.813080
2,750002,47.254486
3,750003,79.497770
4,750004,47.916763
